# Lasso Regression 

## 1. Problem Setup

Assume we have a dataset

$$
\{(x_i,y_i)\}_{i=1}^{N}
$$

where

$$
x_i \in \mathbb{R}^D
$$

is the feature vector and

$$
y_i \in \mathbb{R}
$$

is the target variable.

The linear regression model assumes

$$
y_i = w_0 + \sum_{j=1}^{D} w_j x_{ij} + \epsilon_i
$$

where the noise term follows

$$
\epsilon_i \sim N(0,\sigma^2)
$$

---

# 2. Matrix Representation

Define the design matrix

$$
X =
\begin{bmatrix}
x_{11} & x_{12} & \dots & x_{1D} \\
x_{21} & x_{22} & \dots & x_{2D} \\
\vdots & \vdots & \dots & \vdots \\
x_{N1} & x_{N2} & \dots & x_{ND}
\end{bmatrix}
$$

and the target vector

$$
y =
\begin{bmatrix}
y_1 \\
y_2 \\
\vdots \\
y_N
\end{bmatrix}
$$

The linear model becomes

$$
y = Xw + \epsilon
$$

---

# 3. Adding the Bias Term

To incorporate the intercept term we augment the design matrix with a column of ones

$$
X =
\begin{bmatrix}
1 & x_{11} & x_{12} & \dots & x_{1D} \\
1 & x_{21} & x_{22} & \dots & x_{2D} \\
\vdots & \vdots & \vdots & \dots & \vdots \\
1 & x_{N1} & x_{N2} & \dots & x_{ND}
\end{bmatrix}
$$

The weight vector becomes

$$
w =
\begin{bmatrix}
w_0 \\
w_1 \\
\vdots \\
w_D
\end{bmatrix}
$$

---



# 4. Lasso Objective Function

Lasso introduces an $L_1$ regularization penalty

$$
||w||_1 =
\sum_{j=1}^{D}|w_j|
$$

The optimization problem becomes

$$
\min_w
\left[
\frac{1}{2}
||y - Xw||_2^2
+
\lambda
\sum_{j=1}^{D}|w_j|
\right]
$$

where

$$
\lambda \ge 0
$$

controls the strength of regularization.

The intercept $w_0$ is not penalized.

---

# 5. Non-Differentiability of the L1 Norm

The function

$$
|w|
$$

is not differentiable at

$$
w = 0
$$

Therefore the Lasso objective cannot be solved using the closed form solution of ordinary least squares.

Instead we use an iterative optimization technique called **coordinate descent**.

---

# 6. Coordinate Descent Optimization

Coordinate descent updates one coefficient at a time while keeping the remaining coefficients fixed.

Let

$$
w = (w_0, w_1, w_2, \dots, w_D)
$$

At each step we solve the optimization problem for a single coordinate

$$
w_j
$$

while holding all other weights constant.

---

# 7. Residual Definition

Define the residual vector

$$
r = y - Xw
$$

This vector represents the prediction error.

Initially the weights are

$$
w = 0
$$

which implies

$$
r = y
$$

---

# 8. Partial Residual for Feature $j$

When updating coefficient $w_j$, we isolate the contribution of feature $j$.

Define the partial residual

$$
r_i^{(j)} =
y_i -
\sum_{k \ne j} x_{ik} w_k
$$

This removes the contribution of feature $j$ from the prediction.

---

# 9. Computing $\rho_j$

The coordinate descent update requires computing

$$
\rho_j =
\sum_{i=1}^{N} x_{ij} r_i^{(j)}
$$

In vector form

$$
\rho_j = x_j^T r
$$

where

$$
x_j
$$

is the $j^{th}$ column of the design matrix.

---

# 10. Feature Norm

For each feature we compute

$$
||x_j||^2 =
\sum_{i=1}^{N} x_{ij}^2
$$

This value is reused in the coordinate updates.

---

# 11. Soft Thresholding Operator

The update rule for Lasso regression uses the soft threshold operator

$$
S(\rho,\lambda)
$$

The weight update becomes

$$
w_j =
\frac{S(\rho_j,\lambda)}{\sum_{i=1}^{N}x_{ij}^2}
$$

---

# 12. Soft Threshold Function

The soft threshold function is defined as

$$
S(\rho,\lambda) =
\begin{cases}
\rho - \lambda & \text{if } \rho > \lambda \\
\rho + \lambda & \text{if } \rho < -\lambda \\
0 & \text{if } |\rho| \le \lambda
\end{cases}
$$

This operator shrinks coefficients toward zero.

If

$$
|\rho| \le \lambda
$$

then

$$
w_j = 0
$$

which produces sparse models.

---

# 13. Intercept Update (Bias Term)

The intercept term $w_0$ is **not regularized** in Lasso regression.  
Therefore its update does **not include the $L_1$ penalty** and reduces to a standard least–squares update.


## Optimization for the Intercept

When updating the intercept while keeping all other coefficients fixed, the objective reduces to

$$
\min_{w_0}
\frac{1}{2}
\sum_{i=1}^{N}
\left(
r_i - x_{i0} w_0
\right)^2
$$

where

$$
r_i = y_i - \sum_{j \ne 0} x_{ij} w_j
$$

is the **partial residual excluding the intercept term**.



## General Coordinate Descent Update

The coordinate descent update for a coefficient $w_j$ without regularization is

$$
w_j =
\frac{x_j^T r}{x_j^T x_j}
$$

Applying this to the intercept term gives

$$
w_0 =
\frac{x_0^T r}{x_0^T x_0}
$$



## Structure of the Bias Feature

The intercept feature column is a vector of ones

$$
x_0 =
\begin{bmatrix}
1 \\
1 \\
\vdots \\
1
\end{bmatrix}
\in \mathbb{R}^N
$$

Therefore

$$
x_0^T r =
\sum_{i=1}^{N} r_i
$$

and

$$
x_0^T x_0 =
\sum_{i=1}^{N} 1^2 = N
$$



## Final Intercept Update

Substituting these values into the update formula gives

$$
w_0 =
\frac{\sum_{i=1}^{N} r_i}{N}
$$

which simplifies to

$$
\boxed{
w_0 =
\frac{1}{N}
\sum_{i=1}^{N} r_i
}
$$

Thus the intercept is simply the **mean of the residuals**.

---
# 14. Efficient Residual Update

Instead of recomputing

$$
r = y - Xw
$$

after every update, the residual is updated incrementally.

Before updating the coefficient

$$
r = r + x_j w_j
$$

After computing the new value

$$
r = r - x_j w_j
$$

This ensures that the residual always satisfies

$$
r = y - Xw
$$

while avoiding expensive recomputation.

---

# 15. Convergence Criterion

The coordinate descent algorithm stops when the change in coefficients becomes sufficiently small.

The stopping condition is

$$
\max_j |w_j^{(t)} - w_j^{(t-1)}| < \epsilon
$$

where

$$
\epsilon
$$

is a small tolerance value.

---

# 16. Prediction

After training, predictions are computed using

$$
\hat{y} = Xw
$$

For a single observation

$$
\hat{y}_i =
w_0 +
\sum_{j=1}^{D}
w_j x_{ij}
$$

---

# 17. Algorithm Summary

Initialize

$$
w = 0
$$

Repeat until convergence:

For each feature $j$

1. Compute

$$
\rho_j = x_j^T r
$$

2. Apply soft thresholding

$$
w_j =
\frac{S(\rho_j,\lambda)}{\sum x_{ij}^2}
$$

3. Update the residual

$$
r = r + x_j w_j^{old}
$$

$$
r = r - x_j w_j^{new}
$$

Stop when the coefficients stabilize.

---

# 18. Sparsity Property

The $L_1$ penalty forces some coefficients to become exactly zero.

If

$$
|\rho_j| \le \lambda
$$

then

$$
w_j = 0
$$

Thus Lasso performs automatic feature selection.

---



# 19. Final Optimization Problem

The final optimization problem solved by Lasso regression is

$$
\boxed{
\min_w
\left[
\frac{1}{2}
\sum_{i=1}^{N}
\left(
y_i -
\sum_{j=0}^{D} x_{ij}w_j
\right)^2
+
\lambda
\sum_{j=1}^{D}|w_j|
\right]
}
$$

In [1]:
class Lasso_Regression:
    def __init__(self,lambda_=0.1,add_bias=True,num_iterations=1000,convergence_tol=1e-5):
        
        self.lambda_ = lambda_
        self.add_bias = add_bias
        self.num_iterations = num_iterations
        self.convergence_tol = convergence_tol

        self.weights =None


    def soft_threshold(self,rho):
        
        condition_list = [rho > self.lambda_ , rho < -self.lambda_]
        choice_list = [rho-self.lambda_ , rho +self.lambda_]

        return np.select(condition_list,choice_list,default=0)

    def fit(self,X,y):
        
        X = np.asarray(X)
        y = np.asarray(y)

        y = y.reshape(-1)

        if X.ndim==1:
            X = X.reshape(-1,1)

        N , D = X.shape

        if self.add_bias :
            X = np.hstack((np.ones((N,1)),X))
            D +=1

        self.weights = np.zeros(D)

       
        X_norm_sq = np.sum(X*X,axis=0)
        
        residual = y.copy()

        for i in range(self.num_iterations):
            weights_old = self.weights.copy()

   
            for feature_no in range(D):
                residual = residual + X[:,feature_no] * self.weights[feature_no]
                rho_j = X[:,feature_no].T @ residual
                

                if self.add_bias and feature_no==0:
                    self.weights[feature_no] = np.mean(residual)

                else: 
                    
                    self.weights[feature_no] = self.soft_threshold(rho_j)/X_norm_sq[feature_no]
                    #self.weights[feature_no] = np.sign(rho_j)*np.max(abs(rho_j)-self.lambda_,0)/X_norm_sq[feature_no]

                residual = residual - X[:,feature_no] * self.weights[feature_no]


                
            if i>0 and np.max(np.abs(self.weights-weights_old)) <=self.convergence_tol:
                print(f'Solution Converged in iteration number : {i}')
                break
        print(f'The final weights are : {self.weights}')


    def predict(self,X):
        
        X = np.asarray(X)

        if X.ndim==1:
            X = X.reshape(-1,1)

        N = len(X)

        if self.add_bias :
            X = np.hstack((np.ones((N,1)),X))

        return X @ self.weights     

### 1. Objective

The goal of this experiment is to study the effect of the **regularization parameter** $\lambda$ in **Lasso Regression** on model behavior.


- Impact of $\lambda$ on predictions  
- Effect of including vs excluding **bias term**  
- How increasing $\lambda$ influences model sparsity  

### Dataset

We consider a small synthetic dataset with **8 samples** and **3 features**:

---


In [2]:
import numpy as np

X = np.array([
    [1.2, 3.1, 0.5],
    [2.0, 1.5, 1.2],
    [3.1, 2.2, 0.3],
    [4.0, 3.0, 2.1],
    [5.2, 4.1, 1.8],
    [6.1, 2.9, 2.5],
    [7.0, 3.5, 3.1],
    [8.2, 4.2, 2.9]
])

y = np.array([
    4.1, 3.8, 5.2, 7.5,
    9.1, 9.8, 11.5, 12.7
])

---

In [3]:
lambda_ =[0, 0.001,5,10,100,1000]
for l in lambda_:
    print(f"Model training for lambda = {l} without bias \n")
    model = Lasso_Regression(lambda_=l,add_bias=False)
    model.fit(X,y)
    print(f"Predictions : {model.predict(X)} \n")


Model training for lambda = 0 without bias 

Solution Converged in iteration number : 202
The final weights are : [0.95672747 0.82677614 0.55685668]
Predictions : [ 3.98950734  3.82184718  4.95181968  7.47663735  9.36710706  9.6258301
 11.31706452 12.93250945] 

Model training for lambda = 0.001 without bias 

Solution Converged in iteration number : 202
The final weights are : [0.95690592 0.82672199 0.55646765]
Predictions : [ 3.9893591   3.82165602  4.95213704  7.47637174  9.36711274  9.62578904
 11.31691816 12.93261713] 

Model training for lambda = 5 without bias 

Solution Converged in iteration number : 77
The final weights are : [1.29478343 0.57885403 0.        ]
Predictions : [ 3.3481876   3.45784791  5.2873075   6.91569581  9.10617536  9.57685562
 11.08947312 13.04841106] 

Model training for lambda = 10 without bias 

Solution Converged in iteration number : 72
The final weights are : [1.41046405 0.34020288 0.        ]
Predictions : [ 2.74718578  3.33123241  5.12088488  6.662

---

In [4]:
for l in lambda_:
    print(f"Model training for lambda = {l} with bias \n")
    model = Lasso_Regression(lambda_=l,add_bias=True)
    model.fit(X,y)
    print(f"Predictions : {model.predict(X)} \n")

Model training for lambda = 0 with bias 

Solution Converged in iteration number : 338
The final weights are : [0.3988445  0.97915538 0.68827387 0.52872865]
Predictions : [ 3.97184426  4.02404044  5.10704727  7.49061778  9.26408689  9.68950815
 11.3009495  12.85198193] 

Model training for lambda = 0.001 with bias 

Solution Converged in iteration number : 338
The final weights are : [0.39950599 0.97937096 0.68799014 0.52829295]
Predictions : [ 3.97166704  4.02418465  5.10762215  7.49037543  9.26392185  9.68957261
 11.30077633 12.85195599] 

Model training for lambda = 5 with bias 

Solution Converged in iteration number : 53
The final weights are : [2.26315113 1.23899044 0.         0.        ]
Predictions : [ 3.74993966  4.74113202  6.10402151  7.2191129   8.70590144  9.82099284
 10.93608423 12.42287277] 

Model training for lambda = 10 with bias 

Solution Converged in iteration number : 53
The final weights are : [2.80740333 1.12067485 0.         0.        ]
Predictions : [ 4.152213

---

### 2. Observations

#### Effect of $\lambda$

- **$\lambda = 0$**
  - Equivalent to standard linear regression  
  - No regularization  
  - All features contribute to prediction  

- **Small $\lambda$ (e.g., 0.001)**
  - Slight regularization  
  - Weights shrink slightly but remain mostly non-zero  

- **Moderate $\lambda$ (5, 10)**
  - Some weights become exactly zero  
  - Model starts performing **feature selection**  

- **Large $\lambda$ (100, 1000)**
  - Strong regularization  
  - Most weights shrink to zero  
  - Model becomes very simple
  - Bias term approximates the mean of $y$, causing predictions to converge to $y$’s average

---

#### Effect of Bias Term

- **Without bias**
  - Model is constrained to pass through origin  
  - Predictions may be systematically biased  

- **With bias**
  - Model can shift appropriately  
  - Typically results in better fit  

---

### 3. Key Insight

Lasso Regression performs **automatic feature selection**:

$$
\lambda \uparrow \quad \Rightarrow \quad \|\mathbf{w}\|_0 \downarrow
$$

(i.e., number of non-zero weights decreases)

---

### 4. Conclusion

- $\lambda$ controls the **bias-variance tradeoff**:
  
  - Small $\lambda$ → complex model (low bias, high variance)  
  - Large $\lambda$ → simple model (high bias, low variance)  

- Lasso is useful when:
  - Dataset contains **irrelevant features**
  - Model interpretability is important  

- Including a **bias term** generally improves performance  

$$
\text{Best practice: choose } \lambda \text{ using cross-validation}
$$

---